In [ ]:
# ===== shortest-CoV TOP-3 CoV on ms640 @ budget 100,000 — CONFIG ========
# ===== edit ONLY this cell ===============================================
#
# WHAT THIS RUNS. For each of the 640 ms640 presentations, the top 3 change-of-
# variables candidates by shortest transformed pair (len(r1)+len(r2)) (frozen on disk by Stage A, in
# results/stable_ac/cov/cov_top3/manifest_ms640_len_top3.jsonl) are searched
# in rank order and the presentation STOPS at the first solve. So a
# presentation costs at most 3 x 100,000 = 300,000 nodes, and usually far less.
# Every search writes a full row: solved, nodes_explored, path_length, the move
# path, the pick that produced it, the presentation's running cum_nodes, and the
# plain-greedy reference for the same presentation — so the nodes/path
# comparison needs no join.
#
# THIS IS THE "len" ARM. Open cov_top3_ms640_abel.ipynb in a second Colab
# session and run it there: it ranks the same candidate families by abelianized magnitude
# instead. The two arms cover all 640 presentations each, write separate jsonls
# (the rule is in the filename), and never touch each other's file. RULE is the
# only knob that differs between the two notebooks.
#
# RESTART CONTRACT. Runtime -> Restart, then Run All, continues this run: SETUP
# resets the repo to the latest push, purges the stale experiments.* modules,
# and seeds the local jsonl back from Drive; RUN resumes from it. Mid-run
# hotfixes must be pushed as .py files — a pushed .ipynb does NOT reach an
# already-open Colab notebook.

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "research/w5/stable-ac-escape"   # must match the actual git branch
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push

MOUNT_DRIVE = True           # mirror the results jsonl to Drive every few
                             # minutes + at the end, and seed it BACK on a fresh
                             # VM so resume continues where the last session
                             # stopped. The runner always writes locally
                             # (appending onto the Drive FUSE mount is unsafe).
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx_results/cov_top3"

# --- experiment knobs ------------------------------------------------------
RULE        = "len"       # <-- the only knob that differs between the arms
BUDGET      = 100_000        # PER SEARCH; a presentation costs <= K * this
K           = 3              # ranks per presentation (the manifest is built at 3)
HIGH_SPEEDUP = True          # compact fast solver (~2.9x); result-neutral —
                             # a solved fast search is re-solved by the normal
                             # solver for its path, so every written row is
                             # identical to a slow-mode row and the files resume
                             # across the two modes
CHUNKS      = 1              # this arm is one session; raise it (with
CHUNK_INDEX = None           # CHUNK_INDEX = 1..CHUNKS) only to split ONE arm
                             # across more machines, then run the MERGE cell


In [ ]:
# ==================== SETUP (clone / pull / Drive) ========================
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy pyyaml")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)                      # relative paths + "import experiments…"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# a `git reset --hard` rewrites .py files but sys.modules keeps the OLD module
# objects -- drop them so RUN imports what SETUP just fetched (pull != reload)
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# --- Drive: mount + seed-back (fresh VM -> local resume state) -------------
import glob, shutil
LOCAL_OUT = os.path.join(REPO_ROOT, "results", "stable_ac", "cov", "cov_top3")
os.makedirs(LOCAL_OUT, exist_ok=True)
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for src in glob.glob(os.path.join(DRIVE_DIR, "*.jsonl")):
        dst = os.path.join(LOCAL_OUT, os.path.basename(src))
        # the bigger file wins: local mid-run state beats a stale mirror, and on
        # a fresh VM the mirror beats the (absent/empty) local file
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            shutil.copyfile(src, dst)
            print("seeded from Drive:", os.path.basename(src))


In [ ]:
# ==================== RUN =================================================
# Production budgets run HERE, never on the dev machine: the repo caps any
# locally-launched search at 1,000 nodes, and the runner enforces that cap
# unless this flag is set.
os.environ["ACSOLVERX_ALLOW_BIG"] = "1"

from experiments.stable_ac.cov.run import cov_top3_manifest as manifest
from experiments.stable_ac.cov.run import cov_top3_run as R

# Stage A is committed (1,920 picks over 640 presentations per rule) — rebuild
# only if a shallow clone somehow lacks it. It explores ZERO nodes and takes
# ~3 s. The manifest path is DERIVED from RULE, never passed beside it: a stale
# path plus the other rule's name is a wrong experiment wearing a plausible
# filename, which is also why the runner asserts every manifest row's own rule.
MANIFEST = manifest.manifest_path(RULE)
if not os.path.exists(os.path.join(REPO_ROOT, MANIFEST)):
    print("manifest missing — rebuilding (no search)")
    manifest.build(rule=RULE, out_path=MANIFEST)
groups = manifest.load_manifest(MANIFEST, rule=RULE)
print(f"manifest [{RULE}]: {len(groups)} presentations x <= {K} ranks")

# mirror local jsonls -> Drive every 3 min (and once at the end). Whole-file
# copies of an append-only jsonl: a torn tail line is repaired on resume. The
# thread never prints (a background thread must not).
import threading
def _sync_to_drive():
    if not (IN_COLAB and MOUNT_DRIVE):
        return
    for src in glob.glob(os.path.join(LOCAL_OUT, "*.jsonl")):
        dst = os.path.join(DRIVE_DIR, os.path.basename(src))
        # size-monotonic: an append-only jsonl only ever grows, so never
        # overwrite a bigger Drive copy with a smaller local one (a session
        # holding a stale seeded copy of the OTHER arm must not clobber it)
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            tmp = dst + ".tmp"
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
def _mirror_loop():
    while not _mirror_stop.wait(180):
        try: _sync_to_drive()
        except Exception: pass                 # transient Drive hiccup: next tick
_mirror_stop = threading.Event()
threading.Thread(target=_mirror_loop, daemon=True).start()

try:
    out_path = R.run(rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                     chunk_index=CHUNK_INDEX, high_speedup=HIGH_SPEEDUP,
                     manifest=MANIFEST)
finally:
    _mirror_stop.set()
    _sync_to_drive()                           # final sync, incl. the last rows
    if IN_COLAB and MOUNT_DRIVE:
        print("mirrored to", DRIVE_DIR)

# This arm's score. The plain-greedy controls are read by TRUNCATING the frozen
# 1,000,000-node ms640 baseline (zero new search) at BUDGET and at K x BUDGET,
# and the paired nodes/path comparison runs over the presentations BOTH arms
# solved. Then the gate: every search that overlaps the frozen 10,000-node
# subset-60 sweep must reproduce it node for node.
R.summarize(out_path, rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
            chunk_index=CHUNK_INDEX, manifest=MANIFEST)


In [ ]:
# ============ MERGE / COMPARE (optional, after the other arm finishes) =====
# Separate cell because each half needs a file this session did not write: run
# it only once the run(s) it names have printed their "done" line and mirrored
# to Drive. It re-seeds from Drive first, so run it in whichever session you like.
for src in glob.glob(os.path.join(DRIVE_DIR, "*.jsonl")) if (IN_COLAB and MOUNT_DRIVE) else []:
    dst = os.path.join(LOCAL_OUT, os.path.basename(src))
    if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
        shutil.copyfile(src, dst)

# (a) only if you split THIS arm across several machines (CHUNKS > 1)
if CHUNKS > 1:
    out_path = R.merge_chunks(rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                              manifest=MANIFEST)
    R.summarize(out_path, rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                manifest=MANIFEST)

# (b) the head-to-head, once BOTH arms have finished. Scored on the
# presentations both arms searched — with two sessions finishing at different
# times, an intersection is the only denominator both have earned.
def _arm(rule):
    hits = [p for p in glob.glob(os.path.join(LOCAL_OUT, f"{rule}top{K}_{BUDGET}_*.jsonl"))
            if not R._CHUNK_MARK.search(os.path.basename(p))]
    return max(hits, key=lambda p: sum(1 for _ in open(p))) if hits else None

a, b = _arm("abel"), _arm("len")
print("abel:", a and os.path.basename(a), "| len:", b and os.path.basename(b))
if a and b:
    R.compare_rules(a, b, k=K)
else:
    print("both arms must have a results file here before the head-to-head")
_sync_to_drive()
